# Top2Vec: Keyword Trend Evaluation (Scenario 4)

1. **Ground Truth**: TF-IDF per year → classify keywords as Emerging / Stable / Decaying
2. **SPAN**: Longest consecutive years each keyword appears in model topics

In [1]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
DATA_DIR = Path("../../../../data/preprocess")
TEMPORAL_DIR = Path("../../../../results/top2vec/temporal")
RESULT_DIR = Path("../../../../results/top2vec/tren")

TOP_K = 20  # top keywords per category
EARLY_YEARS = range(2000, 2011)   # 2000-2010
LATE_YEARS = range(2015, 2026)     # 2015-2025

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Data: {DATA_DIR}")
print(f"Topics: {TEMPORAL_DIR}")
print(f"Output: {RESULT_DIR}")
print(f"Top-K keywords per category: {TOP_K}")

Data: ../../../../data/preprocess
Topics: ../../../../results/top2vec/temporal
Output: ../../../../results/top2vec/tren
Top-K keywords per category: 20


## Step 1: Compute TF-IDF per Year
Calculate average TF-IDF score for each word in each year.

In [3]:
def compute_yearly_tfidf(subject):
    df = pd.read_csv(DATA_DIR / subject / "emb/v1.csv")
    df["year"] = pd.to_datetime(df["submitted_date"]).dt.year

    # Parse text: could be string repr of list or plain text
    def to_text(val):
        try:
            tokens = ast.literal_eval(val)
            if isinstance(tokens, list):
                return " ".join(tokens)
        except (ValueError, SyntaxError):
            pass
        return str(val)

    df["text_str"] = df["text"].apply(to_text)
    years = sorted(df["year"].unique())

    # Compute TF-IDF per year
    yearly_scores = {}  # word → {year: avg_tfidf}
    for year in years:
        year_docs = df[df["year"] == year]["text_str"].tolist()
        if len(year_docs) < 5:
            continue

        tfidf = TfidfVectorizer(max_features=5000, min_df=2, stop_words='english')
        matrix = tfidf.fit_transform(year_docs)
        feature_names = tfidf.get_feature_names_out()
        avg_scores = np.asarray(matrix.mean(axis=0)).flatten()

        for word, score in zip(feature_names, avg_scores):
            if word not in yearly_scores:
                yearly_scores[word] = {}
            yearly_scores[word][year] = float(score)

    return yearly_scores, years

# Cache results
tfidf_cache = {}
for subject in LIST_SUBJECT:
    print(f"Computing TF-IDF for {subject}...")
    tfidf_cache[subject] = compute_yearly_tfidf(subject)
    n_words = len(tfidf_cache[subject][0])
    print(f"  {n_words} unique words tracked")

Computing TF-IDF for cs...


  12598 unique words tracked
Computing TF-IDF for math...


  11247 unique words tracked
Computing TF-IDF for physics...


  11442 unique words tracked


## Step 2: Classify Keywords
- **Emerging**: low early (2000-2010), high late (2015-2025)
- **Stable**: consistently high across all years
- **Decaying**: high early, low late

In [4]:
def classify_keywords(yearly_scores, years, top_k=20):
    """
    Classify words using LINEAR REGRESSION SLOPE of TF-IDF over time.
    - Emerging: strong positive slope (growing importance)
    - Stable: near-zero slope + high avg TF-IDF (consistently important)
    - Decaying: strong negative slope (declining importance)
    """
    from scipy.stats import linregress

    word_stats = []
    years_arr = np.array(years, dtype=float)

    for word, scores in yearly_scores.items():
        # Get TF-IDF values aligned with years
        vals = np.array([scores.get(y, 0.0) for y in years])

        # Only consider words present in at least 3 years
        n_present = np.sum(vals > 0)
        if n_present < 3:
            continue

        # Linear regression: TF-IDF = a + slope * year
        slope, intercept, r_val, p_val, std_err = linregress(years_arr, vals)

        overall_avg = vals.mean()
        overall_std = vals.std()

        # First and last year the word appears
        present_years = [y for y, v in zip(years, vals) if v > 0]
        first_year = min(present_years)
        last_year = max(present_years)

        # Early vs late avg (for display)
        early = [y for y in years if y <= 2010]
        late = [y for y in years if y >= 2015]
        early_avg = np.mean([scores.get(y, 0.0) for y in early])
        late_avg = np.mean([scores.get(y, 0.0) for y in late])

        word_stats.append({
            'word': word,
            'slope': slope,
            'r_squared': r_val**2,
            'p_value': p_val,
            'overall_avg': overall_avg,
            'overall_std': overall_std,
            'early_avg': early_avg,
            'late_avg': late_avg,
            'n_present': n_present,
            'first_year': first_year,
            'last_year': last_year,
        })

    stats_df = pd.DataFrame(word_stats)

    # Emerging: highest positive slope (statistically significant)
    emerging_pool = stats_df[stats_df['slope'] > 0]
    emerging = emerging_pool.nlargest(top_k, 'slope')

    # Stable: high avg TF-IDF + lowest absolute slope → consistent importance
    # Score: avg / (1 + |slope| * 10000)  → rewards high avg, penalizes any trend
    used = set(emerging['word'])
    stable_pool = stats_df[~stats_df['word'].isin(used)].copy()
    stable_pool['stability'] = stable_pool['overall_avg'] / (1 + stable_pool['slope'].abs() * 10000)
    stable = stable_pool.nlargest(top_k, 'stability')

    # Decaying: strongest negative slope
    used.update(stable['word'])
    decay_pool = stats_df[(~stats_df['word'].isin(used)) & (stats_df['slope'] < 0)]
    decaying = decay_pool.nsmallest(top_k, 'slope')

    return emerging, stable, decaying, stats_df

# Classify and save
keyword_cache = {}
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Keyword Classification: {subject.upper()}")
    print(f"{'='*70}")

    yearly_scores, years = tfidf_cache[subject]
    emerging, stable, decaying, stats_df = classify_keywords(yearly_scores, years, TOP_K)
    keyword_cache[subject] = (emerging, stable, decaying)

    # Save ground truth
    gt_rows = []
    for cat, cat_df in [('emerging', emerging), ('stable', stable), ('decaying', decaying)]:
        for _, r in cat_df.iterrows():
            gt_rows.append({
                'word': r['word'], 'category': cat,
                'slope': round(r['slope'], 8),
                'r_squared': round(r['r_squared'], 4),
                'early_avg': round(r['early_avg'], 6),
                'late_avg': round(r['late_avg'], 6),
                'overall_avg': round(r['overall_avg'], 6),
                'first_year': int(r['first_year']),
                'n_present': int(r['n_present']),
            })
    gt_df = pd.DataFrame(gt_rows)
    gt_df.to_csv(RESULT_DIR / subject / 'ground_truth_keywords.csv', index=False)

    for cat, icon, cat_df in [
        ('Emerging', chr(0x1F4C8), emerging),
        ('Stable', chr(0x1F512), stable),
        ('Decaying', chr(0x1F4C9), decaying),
    ]:
        print(f"\n  {icon} {cat} (top {TOP_K}):")
        print(f"  {'Word':25s} {'Slope':>10s} {'R²':>6s} {'Early':>8s} {'Late':>8s} {'1st yr':>6s}")
        print(f"  {'-'*70}")
        for _, r in cat_df.head(10).iterrows():
            print(f"  {r['word']:25s} {r['slope']:10.6f} {r['r_squared']:6.3f} "
                  f"{r['early_avg']:8.5f} {r['late_avg']:8.5f} {int(r['first_year']):>6d}")

    print(f"\n  Saved: {RESULT_DIR / subject / 'ground_truth_keywords.csv'}")


Keyword Classification: CS



  📈 Emerging (top 20):
  Word                           Slope     R²    Early     Late 1st yr
  ----------------------------------------------------------------------
  learning                    0.000928  0.725  0.00834  0.02423   2000
  deep                        0.000662  0.687  0.00041  0.01215   2002
  training                    0.000656  0.756  0.00241  0.01286   2000
  models                      0.000630  0.657  0.00928  0.01864   2000
  image                       0.000608  0.911  0.00384  0.01366   2000
  neural                      0.000560  0.616  0.00392  0.01353   2000
  dataset                     0.000544  0.873  0.00055  0.00913   2001
  datasets                    0.000485  0.862  0.00097  0.00866   2000
  detection                   0.000456  0.894  0.00377  0.01080   2000
  tasks                       0.000451  0.763  0.00277  0.00976   2000

  🔒 Stable (top 20):
  Word                           Slope     R²    Early     Late 1st yr
  ---------------------------


  📈 Emerging (top 20):
  Word                           Slope     R²    Early     Late 1st yr
  ----------------------------------------------------------------------
  mathbb                      0.001242  0.902  0.00940  0.02984   2000
  mathcal                     0.000874  0.919  0.00570  0.01976   2000
  mathrm                      0.000393  0.897  0.00086  0.00705   2002
  frac                        0.000357  0.926  0.00336  0.00904   2000
  optimization                0.000347  0.917  0.00101  0.00627   2000
  method                      0.000320  0.894  0.00734  0.01257   2000
  time                        0.000312  0.936  0.00809  0.01287   2000
  optimal                     0.000311  0.968  0.00316  0.00788   2000
  control                     0.000295  0.956  0.00234  0.00694   2000
  numerical                   0.000288  0.949  0.00331  0.00786   2000

  🔒 Stable (top 20):
  Word                           Slope     R²    Early     Late 1st yr
  ---------------------------


  📈 Emerging (top 20):
  Word                           Slope     R²    Early     Late 1st yr
  ----------------------------------------------------------------------
  learning                    0.000403  0.718  0.00142  0.00722   2000
  imaging                     0.000288  0.948  0.00248  0.00686   2000
  neural                      0.000267  0.639  0.00112  0.00492   2000
  based                       0.000261  0.939  0.01104  0.01498   2000
  network                     0.000250  0.461  0.00632  0.00959   2000
  high                        0.000249  0.726  0.01090  0.01497   2000
  materials                   0.000241  0.853  0.00271  0.00630   2000
  demonstrate                 0.000240  0.972  0.00433  0.00799   2000
  performance                 0.000235  0.918  0.00374  0.00731   2000
  networks                    0.000220  0.234  0.00662  0.00916   2000

  🔒 Stable (top 20):
  Word                           Slope     R²    Early     Late 1st yr
  ---------------------------

## Step 3: SPAN Calculation
For each ground truth keyword, check presence in model topics per year.
SPAN = longest consecutive year sequence the keyword appears in any topic.

In [5]:
def compute_span(keyword, topic_words_by_year, years):
    """
    Compute SPAN and topic count per year for a keyword.
    Returns: max_span, total_present, presence, best_start, topic_counts
    """
    presence = []
    topic_counts = []  # number of topics containing this word per year
    for y in years:
        count = sum(1 for words in topic_words_by_year.get(y, []) if keyword in words)
        topic_counts.append(count)
        presence.append(1 if count > 0 else 0)

    max_span = 0
    current_span = 0
    span_start = None
    best_start = None
    for i, p in enumerate(presence):
        if p == 1:
            if current_span == 0:
                span_start = years[i]
            current_span += 1
            if current_span > max_span:
                max_span = current_span
                best_start = span_start
        else:
            current_span = 0

    total_present = sum(presence)
    return max_span, total_present, presence, best_start, topic_counts


for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"SPAN Analysis: {subject.upper()}")
    print(f"{'='*70}")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")
    years = sorted(evo_df['year'].unique())

    topic_words_by_year = defaultdict(list)
    for _, row in evo_df.iterrows():
        words = set(w.strip() for w in str(row['top_words']).split(','))
        topic_words_by_year[int(row['year'])].append(words)

    emerging, stable, decaying = keyword_cache[subject]

    span_rows = []
    for cat, cat_name, cat_df in [
        ('emerging', chr(0x1F4C8) + ' Emerging', emerging),
        ('stable', chr(0x1F512) + ' Stable', stable),
        ('decaying', chr(0x1F4C9) + ' Decaying', decaying),
    ]:
        print(f"\n  {cat_name}:")
        # Header with year labels
        yr_labels = ''.join([str(y)[-2:] for y in years])
        print(f"  {'Keyword':20s} {'SPAN':>4} {'Tot':>3}  Topics/year (count per year)")
        print(f"  {'-'*70}")

        for _, r in cat_df.iterrows():
            word = r['word']
            max_span, total, presence, best_start, topic_counts = compute_span(
                word, topic_words_by_year, years
            )

            # Visual: show topic count per year (0=dot, 1-9=number, 10+=+)
            count_str = ''
            for c in topic_counts:
                if c == 0:
                    count_str += chr(0x00B7)  # middle dot
                elif c <= 9:
                    count_str += str(c)
                else:
                    count_str += '+'

            total_topics = sum(topic_counts)

            span_rows.append({
                'subject': subject, 'word': word, 'category': cat,
                'span': max_span, 'total_years_present': total,
                'span_start': best_start if best_start else -1,
                'total_years': len(years),
                'coverage_pct': round(total / len(years) * 100, 1),
                'total_topic_hits': total_topics,
                'avg_topics_when_present': round(total_topics / total, 2) if total > 0 else 0,
                'topic_counts_per_year': str(topic_counts),
                'presence': ''.join([chr(0x2588) if p else chr(0x00B7) for p in presence]),
            })

            avg_t = f"{total_topics/total:.1f}" if total > 0 else "0"
            print(f"  {word:20s} {max_span:4d} {total:3d}  {count_str}  (avg {avg_t} topics)")

    span_df = pd.DataFrame(span_rows)
    span_df.to_csv(RESULT_DIR / subject / 'keyword_span.csv', index=False)
    print(f"\n  Year index: {'  '.join([str(y) for y in years[::5]])}")
    print(f"  Saved: {RESULT_DIR / subject / 'keyword_span.csv'}")


SPAN Analysis: CS

  📈 Emerging:
  Keyword              SPAN Tot  Topics/year (count per year)
  ----------------------------------------------------------------------
  learning               26  26  147333343446++++++++++++++  (avg 26.8 topics)
  deep                   13  14  ··········1··348++++++++66  (avg 13.6 topics)
  training               14  17  ·1·····1··1·131257++++++++  (avg 9.6 topics)
  models                 15  21  1···11212··26447768+++++++  (avg 15.5 topics)
  image                  23  24  1··13253177+++++++++++++++  (avg 15.4 topics)
  neural                 24  24  ··23471312612268++++++++++  (avg 9.7 topics)
  dataset                 7   8  ··············1····4424239  (avg 3.6 topics)
  datasets                4  10  ········1·····11·1·11·2153  (avg 1.7 topics)
  detection              22  25  211·3111333435558+++++++++  (avg 8.0 topics)
  tasks                  10  15  ··1······1··121·2226369784  (avg 3.7 topics)
  images                 21  23  ·1·2·121336935


  📈 Emerging:
  Keyword              SPAN Tot  Topics/year (count per year)
  ----------------------------------------------------------------------
  mathbb                 22  25  113·24534+99++++++++++++++  (avg 33.6 topics)
  mathcal                26  26  213129646+5+9+++++++++++++  (avg 19.9 topics)
  mathrm                 14  18  ·11··2·1····225795575++9++  (avg 6.7 topics)
  frac                   25  25  ·12134611725447+++++++++++  (avg 7.9 topics)
  optimization           21  23  1··3·11111225343343787+689  (avg 4.0 topics)
  method                 26  26  1211111322327678875895+688  (avg 4.7 topics)
  time                   26  26  2533333365738+6+++++++++++  (avg 8.8 topics)
  optimal                21  23  ··12·122233433468663976664  (avg 4.2 topics)
  control                26  26  11221253535554645576666755  (avg 4.3 topics)
  numerical              23  23  ···12212313232455555675565  (avg 3.7 topics)
  convergence            16  21  ··11·222··1215365543646534  (avg 3.


  📈 Emerging:
  Keyword              SPAN Tot  Topics/year (count per year)
  ----------------------------------------------------------------------
  learning               18  24  111·121·2112122338++++++++  (avg 5.3 topics)
  imaging                19  22  ··11·2·1333233455767567556  (avg 4.1 topics)
  neural                 17  25  21121111·11211233688988888  (avg 3.8 topics)
  based                   1   4  ··········1··1······1··1··  (avg 1.0 topics)
  network                26  26  2114258456798799+++++++978  (avg 7.5 topics)
  high                    7  13  ··1·2·1·····21·1···1322166  (avg 2.2 topics)
  materials              12  18  1···3·1·11·1··111126463378  (avg 2.8 topics)
  demonstrate             0   0  ··························  (avg 0 topics)
  performance            15  17  ·····1···1·112221111211122  (avg 1.4 topics)
  networks               23  24  2··1329655695+6787++97+788  (avg 6.7 topics)
  machine                13  14  ·1···········1111334485433  (avg 3.0 to

## Step 4: Summary Analysis

In [6]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Summary: {subject.upper()} (Top2Vec)")
    print(f"{'='*70}")

    span_df = pd.read_csv(RESULT_DIR / subject / "keyword_span.csv")

    # Per-category summary
    summary_rows = []
    for cat in ["emerging", "stable", "decaying"]:
        cat_data = span_df[span_df["category"] == cat]
        s = {
            "subject": subject, "category": cat,
            "n_keywords": len(cat_data),
            "avg_span": round(cat_data["span"].mean(), 2),
            "max_span": int(cat_data["span"].max()),
            "min_span": int(cat_data["span"].min()),
            "avg_coverage_pct": round(cat_data["coverage_pct"].mean(), 2),
            "n_never_captured": int((cat_data["span"] == 0).sum()),
            "n_full_span": int((cat_data["span"] == cat_data["total_years"]).sum()),
        }
        summary_rows.append(s)

        icon = {"emerging": "📈", "stable": "🔒", "decaying": "📉"}[cat]
        print(f"\n  {icon} {cat.upper()}:")
        print(f"    Avg SPAN: {s['avg_span']:.1f} years (max={s['max_span']}, min={s['min_span']})")
        print(f"    Avg coverage: {s['avg_coverage_pct']:.1f}%")
        print(f"    Never captured: {s['n_never_captured']}/{s['n_keywords']}")
        print(f"    Full span (all years): {s['n_full_span']}/{s['n_keywords']}")

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(RESULT_DIR / subject / "keyword_span_summary.csv", index=False)

    # Overall model score
    overall_avg_span = span_df["span"].mean()
    overall_coverage = span_df["coverage_pct"].mean()
    n_captured = (span_df["span"] > 0).sum()
    n_total = len(span_df)

    print(f"\n  ─── Overall ───")
    print(f"  Avg SPAN: {overall_avg_span:.2f} / {span_df['total_years'].iloc[0]} years")
    print(f"  Avg coverage: {overall_coverage:.1f}%")
    print(f"  Keywords captured: {n_captured}/{n_total} ({n_captured/n_total:.0%})")
    print(f"  Saved: {RESULT_DIR / subject}")


Summary: CS (Top2Vec)

  📈 EMERGING:
    Avg SPAN: 13.9 years (max=26, min=0)
    Avg coverage: 66.0%
    Never captured: 1/20
    Full span (all years): 2/20

  🔒 STABLE:
    Avg SPAN: 9.0 years (max=26, min=0)
    Avg coverage: 52.7%
    Never captured: 2/20
    Full span (all years): 1/20

  📉 DECAYING:
    Avg SPAN: 12.8 years (max=26, min=1)
    Avg coverage: 70.8%
    Never captured: 0/20
    Full span (all years): 5/20

  ─── Overall ───
  Avg SPAN: 11.90 / 26 years
  Avg coverage: 63.1%
  Keywords captured: 57/60 (95%)
  Saved: ../../../../results/top2vec/tren/cs

Summary: MATH (Top2Vec)

  📈 EMERGING:
    Avg SPAN: 21.6 years (max=26, min=13)
    Avg coverage: 90.0%
    Never captured: 0/20
    Full span (all years): 7/20

  🔒 STABLE:
    Avg SPAN: 11.2 years (max=26, min=0)
    Avg coverage: 49.6%
    Never captured: 2/20
    Full span (all years): 4/20

  📉 DECAYING:
    Avg SPAN: 21.4 years (max=26, min=3)
    Avg coverage: 88.5%
    Never captured: 0/20
    Full span (all

## Step 5: Paper-Faithful avg-SPAN (Gupta et al., 2018)

Following the SPAN metric from *Deep Temporal-Recurrent-Replicated-Softmax* (arXiv:1711.05626v2):

- **keyword-trend**: binary sequence of keyword appearance in **any** discovered topic per year
- **SPAN (Sₖ)**: length of the longest consecutive 1s in keyword-trend
- **v̂ₖ**: total count of keyword k across **all documents in the corpus** (not just topic words)
- **Sₖ^dict = Sₖ / v̂ₖ**: frequency-normalized SPAN per keyword
- **avg-SPAN = (1/||Q̂||) × Σ Sₖ^dict**: averaged over all unique topic-terms

This computes SPAN over **all** words that appear in discovered topics, normalized by
their corpus frequency to reward models that capture rare-but-trending terms.

In [7]:
def compute_paper_span(keyword, topic_words_by_year, years):
    """
    Compute keyword-trend and SPAN per the paper's definition.
    Returns: span, keyword_trend (list of 0/1)
    """
    trend = []
    for y in years:
        found = any(keyword in words for words in topic_words_by_year.get(y, []))
        trend.append(1 if found else 0)

    # SPAN = longest consecutive 1s
    max_span = 0
    current = 0
    for t in trend:
        if t == 1:
            current += 1
            max_span = max(max_span, current)
        else:
            current = 0

    return max_span, trend


def compute_corpus_word_freq(subject):
    """
    Compute v̂ₖ = total count of each word across ALL documents in the corpus.
    Per paper: v̂ₖ = Σ_{t=1}^{T} Σ_{j=1}^{D_t} v_{j,t}^k
    """
    df = pd.read_csv(DATA_DIR / subject / "bow/v1.csv")
    word_freq = defaultdict(int)
    for text_val in df["text"]:
        try:
            tokens = ast.literal_eval(text_val)
            if isinstance(tokens, list):
                for w in tokens:
                    word_freq[w] += 1
        except (ValueError, SyntaxError):
            for w in str(text_val).split():
                word_freq[w] += 1
    return word_freq


for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Paper-Faithful avg-SPAN: {subject.upper()} (Top2Vec)")
    print(f"{'='*70}")

    # 1. Load topic-word evolution
    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")
    years = sorted(evo_df['year'].unique())

    topic_words_by_year = defaultdict(list)
    for _, row in evo_df.iterrows():
        words = set(w.strip() for w in str(row['top_words']).split(','))
        topic_words_by_year[int(row['year'])].append(words)

    # 2. Collect ALL unique topic-terms (||Q̂||)
    all_topic_terms = set()
    for year_words_list in topic_words_by_year.values():
        for word_set in year_words_list:
            all_topic_terms.update(word_set)

    # 3. Compute corpus word frequency (v̂ₖ from documents)
    print(f"  Computing corpus word frequencies...")
    corpus_freq = compute_corpus_word_freq(subject)
    print(f"  Corpus vocabulary: {len(corpus_freq)} unique words")
    print(f"  Topic-terms (||Q̂||): {len(all_topic_terms)}")

    # 4. Compute SPAN and Sₖ^dict for every topic-term
    paper_rows = []
    for word in sorted(all_topic_terms):
        span, trend = compute_paper_span(word, topic_words_by_year, years)
        v_hat = corpus_freq.get(word, 0)  # corpus frequency
        s_dict = span / v_hat if v_hat > 0 else 0.0

        paper_rows.append({
            'word': word,
            'span': span,
            'v_hat': v_hat,
            's_dict': round(s_dict, 6),
            'keyword_trend': str(trend),
            'total_years': len(years),
            'years_present': sum(trend),
            'coverage_pct': round(sum(trend) / len(years) * 100, 1),
        })

    paper_df = pd.DataFrame(paper_rows)
    paper_df.to_csv(RESULT_DIR / subject / 'keyword_span_paper.csv', index=False)

    # 5. avg-SPAN (paper formula): (1/||Q̂||) × Σ Sₖ^dict
    q_hat = len(all_topic_terms)
    sum_s_dict = paper_df['s_dict'].sum()
    avg_span_paper = sum_s_dict / q_hat if q_hat > 0 else 0.0
    avg_span_simple = paper_df['span'].mean()

    summary = {
        'subject': subject,
        'model': 'Top2Vec',
        'total_unique_terms': q_hat,
        'avg_span_paper': round(avg_span_paper, 6),
        'avg_span_simple': round(avg_span_simple, 4),
        'sum_s_dict': round(sum_s_dict, 6),
        'total_corpus_freq': int(paper_df['v_hat'].sum()),
        'terms_not_in_corpus': int((paper_df['v_hat'] == 0).sum()),
    }
    pd.DataFrame([summary]).to_csv(RESULT_DIR / subject / 'keyword_span_paper_summary.csv', index=False)

    print(f"  avg-SPAN (paper, freq-normalized):  {avg_span_paper:.6f}")
    print(f"  avg-SPAN (simple mean):             {avg_span_simple:.4f}")
    print(f"  Total corpus freq (Σv̂ₖ):            {int(paper_df['v_hat'].sum())}")
    print(f"  Topic-terms not in corpus:          {(paper_df['v_hat'] == 0).sum()}")

    # Top-10 by SPAN
    print(f"\n  Top 10 by SPAN:")
    print(f"  {'Word':<25s} {'SPAN':>5s} {'v̂ₖ':>8s} {'Sₖ^dict':>10s}")
    for _, r in paper_df.nlargest(10, 'span').iterrows():
        print(f"  {r['word']:<25s} {r['span']:>5d} {r['v_hat']:>8d} {r['s_dict']:>10.6f}")

    # Top-10 by Sₖ^dict (highest frequency-normalized SPAN — rare but persistent)
    print(f"\n  Top 10 by Sₖ^dict (rare but persistent):")
    for _, r in paper_df[paper_df['v_hat'] > 0].nlargest(10, 's_dict').iterrows():
        print(f"  {r['word']:<25s} {r['span']:>5d} {r['v_hat']:>8d} {r['s_dict']:>10.6f}")

    print(f"\n  Saved: {RESULT_DIR / subject}")



Paper-Faithful avg-SPAN: CS (Top2Vec)


  Computing corpus word frequencies...


  Corpus vocabulary: 146603 unique words
  Topic-terms (||Q̂||): 14628


  avg-SPAN (paper, freq-normalized):  0.152685
  avg-SPAN (simple mean):             1.6981
  Total corpus freq (Σv̂ₖ):            8020056
  Topic-terms not in corpus:          2612

  Top 10 by SPAN:
  Word                       SPAN      v̂ₖ    Sₖ^dict
  algorithm                    26    81182   0.000320
  automata                     26     1624   0.016010
  channel                      26    19171   0.001356
  clustering                   26     3554   0.007316
  codes                        26       50   0.520000
  data                         26        0   0.000000
  distributed                  26       36   0.722222
  evolutionary                 26     2336   0.011130
  game                         26    12023   0.002163
  graphs                       26       76   0.342105

  Top 10 by Sₖ^dict (rare but persistent):
  images                       21        1  21.000000
  queries                      19        1  19.000000
  channels                     17        1  17.000000

  Corpus vocabulary: 100004 unique words
  Topic-terms (||Q̂||): 12082


  avg-SPAN (paper, freq-normalized):  0.153467
  avg-SPAN (simple mean):             1.9362
  Total corpus freq (Σv̂ₖ):            5231011
  Topic-terms not in corpus:          2317

  Top 10 by SPAN:
  Word                       SPAN      v̂ₖ    Sₖ^dict
  abelian                      26     5591   0.004650
  algebra                      26    26798   0.000970
  algebras                     26     6913   0.003761
  algorithm                    26    30058   0.000865
  banach                       26     1530   0.016993
  boundary                     26    16704   0.001557
  braid                        26     1469   0.017699
  brownian                     26      712   0.036517
  bundles                      26        0   0.000000
  category                     26    14904   0.001744

  Top 10 by Sₖ^dict (rare but persistent):
  lattices                     19        1  19.000000
  distributions                18        1  18.000000
  links                        17        1  17.000000

  Corpus vocabulary: 122859 unique words
  Topic-terms (||Q̂||): 13847


  avg-SPAN (paper, freq-normalized):  0.143720
  avg-SPAN (simple mean):             1.7810
  Total corpus freq (Σv̂ₖ):            6601808
  Topic-terms not in corpus:          2383

  Top 10 by SPAN:
  Word                       SPAN      v̂ₖ    Sₖ^dict
  atom                         26    18876   0.001377
  atomic                       26    12133   0.002143
  atoms                        26        8   3.250000
  beam                         26    27484   0.000946
  bose                         26      220   0.118182
  bunch                        26     2318   0.011217
  calorimeter                  26     1230   0.021138
  cavity                       26    12578   0.002067
  clusters                     26       31   0.838710
  cosmic                       26      955   0.027225

  Top 10 by Sₖ^dict (rare but persistent):
  ions                         26        1  26.000000
  cells                        20        1  20.000000
  flows                        16        1  16.000000